# BestBuy Pipeline v2 - Optimized

So voi buoc1.ipynb:
- Step 2+3 gop lai: phat hien sale -> scrape features ngay (1 vong lap)
- Step 4: Cerebras (OpenRouter) thay GPT-5-mini (22.9s -> ?s)

Pipeline:
1. Search BestBuy (curl_cffi) -> skuIds
2. Filter + Scrape cung luc (priceBlocks + v2 API)
3. Select top 5 deals (Cerebras via LiteLLM)
4. Estimate prices (EnsembleAgent)
5. Final results

In [1]:
# Cell 1: Setup
import os, sys, logging, time

os.chdir('/home/hieu0606sunny/price2026wsl/tech2ai/segment4')
sys.path.insert(0, '/home/hieu0606sunny/price2026wsl/tech2ai/segment4')
sys.path.insert(0, 'base/fix_bestbuy_tocdo_thang3')

logging.basicConfig(level=logging.INFO)
logging.getLogger().setLevel(logging.INFO)

from dotenv import load_dotenv
load_dotenv(override=True)

print(f'Working directory: {os.getcwd()}')
print(f'OPENROUTER_API_KEY: {"SET" if os.getenv("OPENROUTER_API_KEY") else "MISSING"}')
print('Setup complete!')

Working directory: /home/hieu0606sunny/price2026wsl/tech2ai/segment4
OPENROUTER_API_KEY: SET
Setup complete!


In [2]:
# Cell 2: Imports
import chromadb
from litellm import completion
from curl_cffi import requests as curl_requests

from price_agents.bestbuy_deals import ScrapedBestBuyDeal
from price_agents.deals import Deal, DealSelection, Opportunity
from price_agents.ensemble_agent import EnsembleAgent
from buoc1 import search_bestbuy, get_price_blocks, get_product_details

print('Imports done!')

INFO:datasets:PyTorch version 2.9.0 available.


Imports done!


In [3]:
# Cell 3: Init EnsembleAgent (run once)
print('Initializing EnsembleAgent...')
client = chromadb.PersistentClient(path='products_vectorstore')
collection = client.get_or_create_collection('products')
print(f'ChromaDB: {collection.count()} documents')

ensemble = EnsembleAgent(collection)
print('EnsembleAgent ready!')

INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


Initializing EnsembleAgent...


INFO:root:[Ensemble Agent] Initializing Ensemble Agent
INFO:root:[Specialist Agent] Specialist Agent is initializing - connecting to modal
INFO:root:[Specialist Agent] Specialist Agent is ready
INFO:root:[Frontier Agent] Initializing Frontier Agent
INFO:root:[Frontier Agent] Frontier Agent is setting up with OpenAI


ChromaDB: 800000 documents


INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cuda:0
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO:root:[Frontier Agent] Frontier Agent is ready
INFO:root:[Neural Network Agent] Neural Network Agent is initializing
INFO:root:Neural Network is using cuda
INFO:root:[Neural Network Agent] Neural Network Agent is ready and weights are loaded
INFO:root:[Ensemble Agent] Ensemble Agent is ready


EnsembleAgent ready!


In [4]:
# Cell 4: User Input
TEST_KEYWORD = 'laptop'
print(f'Keyword: \'{TEST_KEYWORD}\'')

Keyword: 'laptop'


In [5]:
# Cell 5: Step 1 - Search BestBuy
print('=' * 60)
print(f'STEP 1: Search BestBuy for \'{TEST_KEYWORD}\'')
print('=' * 60)

start = time.time()
session = curl_requests.Session(impersonate='chrome')
session.get('https://www.bestbuy.com/?intl=nosplash', timeout=15)
apollo_products = search_bestbuy(session, TEST_KEYWORD)
print(f'Time: {time.time() - start:.1f}s')

STEP 1: Search BestBuy for 'laptop'


INFO:buoc1:[Step 1] Search: https://www.bestbuy.com/site/searchpage.jsp?st=laptop
INFO:buoc1:  Status: 200 | Size: 1,876,993 bytes
INFO:buoc1:  Found 136 unique SKUs (6 with URLs)


Time: 4.0s


In [6]:
# Cell 6: Step 2+3 - Filter sale + Scrape details (combined)
# priceBlocks tra ve price+onSale, neu onSale -> goi v2 API lay features+URL ngay
print('=' * 60)
print(f'STEP 2+3: Filter sale + Scrape (combined)')
print('=' * 60)

start = time.time()

# Batch price check
sku_ids = [p['skuId'] for p in apollo_products]
price_data = get_price_blocks(session, sku_ids)

# Filter on sale -> immediately scrape features+URL
scraped_deals = []
for sku, pd in price_data.items():
    if not pd['onSale']:
        continue
    
    # On sale -> scrape details immediately
    details = get_product_details(session, sku)
    
    deal = ScrapedBestBuyDeal(
        title=pd['name'],
        brand=pd['brand'],
        price=pd['currentPrice'],
        features=details['features'] or pd['name'],
        url=details['url'] or f'https://www.bestbuy.com/site/{sku}.p',
    )
    scraped_deals.append(deal)
    print(f'  SALE: ${pd["currentPrice"]} (save ${pd["savingsAmount"]}) | {pd["name"][:60]}...')

    if len(scraped_deals) >= 10:
        break

elapsed = time.time() - start
print(f'\nFound {len(scraped_deals)} sale products with features in {elapsed:.1f}s')

INFO:buoc1:[Step 2] priceBlocks: 136 SKUs


STEP 2+3: Filter sale + Scrape (combined)


INFO:buoc1:  Status: 200
INFO:buoc1:  Got price data for 13/136 SKUs


  SALE: $579.99 (save $220.0) | HP - 15.6" Full HD Touch-Screen Laptop - Intel Core i7 - 16G...
  SALE: $299.99 (save $230.0) | Lenovo - IdeaPad 1 15.6" Full HD Laptop - AMD Ryzen 5 7520U ...
  SALE: $179.99 (save $40.0) | HP - 14" Laptop - Intel Processor N150 - 4GB Memory - 128GB ...
  SALE: $779.99 (save $320.0) | Dell - Plus 2-in-1 16" 2K Touch Screen Laptop - Intel Core U...
  SALE: $299.99 (save $129.01) | HP - 15.6" Chromebook Laptop - Intel Processor N200 - 8GB Me...
  SALE: $799.99 (save $210.0) | HP - Victus 15.6" 144Hz Full HD Gaming Laptop - AMD Ryzen 7 ...
  SALE: $679.99 (save $270.0) | Lenovo - Yoga 7 2-in-1 16" 2K Touchscreen Laptop - AMD Ryzen...

Found 7 sale products with features in 4.0s


In [8]:
# Cell 7: Step 4 - Select top 5 deals (Cerebras via OpenRouter)
# Thay GPT-5-mini (22.9s) bang Cerebras
import time
import litellm

print('=' * 60)
print('STEP 4: Select top 5 deals (Cerebras via OpenRouter)')
print('=' * 60)

CEREBRAS_MODEL = 'openrouter/openai/gpt-oss-120b'

SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description.
Most important is that you respond with the 5 deals that have the most detailed product description with price.

**IMPORTANT:**
1. Focus on the product features and specifications, not sales terms.
2. The product_description should be a 3-4 sentence summary of the product itself.
3. Price must be greater than 0.
4. Keep the original URL exactly as provided."""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.

Deals:

"""

# Build prompt
# Lưu ý: Biến 'scraped_deals' và class 'DealSelection' cần phải được định nghĩa trước đó
valid_deals = [d for d in scraped_deals if d.price > 0]
user_prompt = USER_PROMPT_PREFIX
user_prompt += '\n\n'.join([d.describe() for d in valid_deals])
user_prompt += '\n\nInclude up to 5 deals, no more.'

messages = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': user_prompt},
]

start = time.time()
print(f'Calling {CEREBRAS_MODEL} with {len(valid_deals)} deals...')

response = await litellm.acompletion(
    model=CEREBRAS_MODEL,
    messages=messages,
    extra_body={
        'response_format': {
            'type': 'json_schema',
            'json_schema': {
                'name': 'deal_selection',
                'strict': True,
                'schema': DealSelection.model_json_schema(),
            },
        },
        'provider': {
            'order': ['Cerebras'],
            'allow_fallbacks': True,
        },
    },
)

deal_selection = DealSelection.model_validate_json(response.choices[0].message.content)
deal_selection.deals = [d for d in deal_selection.deals if d.price > 0]

elapsed = time.time() - start
print(f'\nSelected {len(deal_selection.deals)} deals in {elapsed:.1f}s:')
for i, deal in enumerate(deal_selection.deals, 1):
    print(f'  [{i}] ${deal.price} | {deal.product_description[:80]}...')

22:25:35 - LiteLLM:INFO: utils.py:3421 - 
LiteLLM completion() model= openai/gpt-oss-120b; provider = openrouter
INFO:LiteLLM:
LiteLLM completion() model= openai/gpt-oss-120b; provider = openrouter


STEP 4: Select top 5 deals (Cerebras via OpenRouter)
Calling openrouter/openai/gpt-oss-120b with 7 deals...

Selected 5 deals in 16.6s:
  [1] $579.99 | The HP 15.6" Full HD Touchscreen laptop is powered by a 13th‑generation Intel Co...
  [2] $779.99 | Dell's Plus 2‑in‑1 16" laptop combines a premium 2K touchscreen with the versati...
  [3] $799.99 | The HP Victus 15.6" gaming laptop is equipped with an AMD Ryzen 7 7445HS process...
  [4] $679.99 | Lenovo Yoga 7 is a 16" 2K IPS LCD touchscreen convertible that offers four modes...
  [5] $299.99 | Lenovo IdeaPad 1 is a budget‑friendly 15.6" Full HD laptop powered by an AMD Ryz...


In [9]:
# Cell 8: Step 5 - Estimate prices (EnsembleAgent)
print('=' * 60)
print('STEP 5: Estimate prices with EnsembleAgent')
print('=' * 60)

start = time.time()
opportunities = []

for i, deal in enumerate(deal_selection.deals, 1):
    print(f'\n[{i}/{len(deal_selection.deals)}] {deal.product_description[:50]}...')
    estimate = ensemble.price(deal.product_description)
    discount = estimate - deal.price
    opportunities.append(Opportunity(deal=deal, estimate=estimate, discount=discount))
    print(f'  Sale: ${deal.price:.2f} | Est: ${estimate:.2f} | Discount: ${discount:.2f}')

print(f'\nEstimation done in {time.time() - start:.1f}s')

INFO:root:[Ensemble Agent] Running Ensemble Agent - preprocessing text
22:26:06 - LiteLLM:INFO: utils.py:3421 - 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
INFO:LiteLLM:
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq


STEP 5: Estimate prices with EnsembleAgent

[1/5] The HP 15.6" Full HD Touchscreen laptop is powered...


22:26:07 - LiteLLM:INFO: utils.py:1302 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Ensemble Agent] Pre-processed text using groq/openai/gpt-oss-20b
INFO:root:[Specialist Agent] Specialist Agent is calling remote fine-tuned model
INFO:root:[Specialist Agent] Specialist Agent completed - predicting $799.00
INFO:root:[Frontier Agent] Frontier Agent is performing a RAG search of the Chroma datastore to find 5 similar products


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5.1 with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $749.00
INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $765.61
INFO:root:[Ensemble Agent] Ensemble Agent complete - returning $755.66
INFO:root:[Ensemble Agent] Running Ensemble Agent - preprocessing text
22:26:44 - LiteLLM:INFO: utils.py:3421 - 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
INFO:LiteLLM:
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq


...

  Sale: $679.99 | Est: $919.70 | Discount: $239.71

[5/5] Lenovo IdeaPad 1 is a budget‑friendly 15.6" Full H...


22:26:53 - LiteLLM:INFO: utils.py:1302 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Ensemble Agent] Pre-processed text using groq/openai/gpt-oss-20b
INFO:root:[Specialist Agent] Specialist Agent is calling remote fine-tuned model
INFO:root:[Specialist Agent] Specialist Agent completed - predicting $299.00
INFO:root:[Frontier Agent] Frontier Agent is performing a RAG search of the Chroma datastore to find 5 similar products


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5.1 with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $449.00
INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $417.76
INFO:root:[Ensemble Agent] Ensemble Agent complete - returning $430.88


  Sale: $299.99 | Est: $430.88 | Discount: $130.89

Estimation done in 49.7s


In [10]:
# Cell 9: Final Results
print('=' * 80)
print('FINAL RESULTS - Sorted by Discount')
print('=' * 80)
print(f'Keyword: \'{TEST_KEYWORD}\'\n')

opportunities.sort(key=lambda x: x.discount, reverse=True)

for i, opp in enumerate(opportunities, 1):
    pct = (opp.discount / opp.estimate * 100) if opp.estimate > 0 else 0
    status = 'HOT DEAL' if opp.discount > 200 else 'Good Deal' if opp.discount > 100 else 'OK' if opp.discount > 0 else 'Overpriced'
    
    print(f'--- #{i} [{status}] ---')
    print(f'  Product:  {opp.deal.product_description[:80]}...')
    print(f'  Sale:     ${opp.deal.price:.2f}')
    print(f'  Estimate: ${opp.estimate:.2f}')
    print(f'  Discount: ${opp.discount:.2f} ({pct:.0f}%)')
    print(f'  URL:      {opp.deal.url[:80]}')
    print()

FINAL RESULTS - Sorted by Discount
Keyword: 'laptop'

--- #1 [HOT DEAL] ---
  Product:  Dell's Plus 2‑in‑1 16" laptop combines a premium 2K touchscreen with the versati...
  Sale:     $779.99
  Estimate: $1099.24
  Discount: $319.25 (29%)
  URL:      https://www.bestbuy.com/product/dell-plus-copilot-pc-16-2k-2-in-1-touchscreen-la

--- #2 [HOT DEAL] ---
  Product:  Lenovo Yoga 7 is a 16" 2K IPS LCD touchscreen convertible that offers four modes...
  Sale:     $679.99
  Estimate: $919.70
  Discount: $239.71 (26%)
  URL:      https://www.bestbuy.com/product/lenovo-yoga-7-2-in-1-16-2k-touchscreen-laptop-am

--- #3 [Good Deal] ---
  Product:  The HP 15.6" Full HD Touchscreen laptop is powered by a 13th‑generation Intel Co...
  Sale:     $579.99
  Estimate: $755.66
  Discount: $175.67 (23%)
  URL:      https://www.bestbuy.com/product/hp-15-6-full-hd-touch-screen-laptop-intel-core-i

--- #4 [Good Deal] ---
  Product:  The HP Victus 15.6" gaming laptop is equipped with an AMD Ryzen 7 7445HS pr